# Loading Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk import word_tokenize, pos_tag, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
import string
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

#nltk.download("punkt_tab")
#nltk.download("stopwords")
#nltk.download("wordnet")
#nltk.download("averaged_perceptron_tagger_eng")


# Creating Dataset

In [2]:
# Loading both data sets and preparing them
sep = "\n-----------------------------------------------------------\n"
arabic_df = pd.read_csv("/kaggle/input/datasets/abedkhooli/arabic-100k-reviews/ar_reviews_100k.tsv", sep="\t")
english_df = pd.read_csv("/kaggle/input/datasets/abdelmalekeladjelet/sentiment-analysis-dataset/sentiment_data.csv")
print("Arabic data length:", len(arabic_df))
print("English data length:",len(english_df))
print("Arabic sample data\n",arabic_df.head())
print("English sample data\n",english_df.head())


#Preparing english_df to be in a standard structure
english_df["text"] = english_df["Comment"].apply(str)
english_df = english_df.drop(columns=["Unnamed: 0","Comment","Sentiment"])
english_df = english_df.sample(frac=1).reset_index(drop=True)[:len(arabic_df)]


# Preaparing arabic_df to be in a standard structure
arabic_df  = arabic_df.drop(columns=["label"])
print(sep)
print("After Preparing\n")
print("Arabic data length:", len(arabic_df))
print("English data length:",len(english_df))
print("Arabic sample data\n",arabic_df.head())
print("English sample data\n",english_df.head())



data = pd.concat([arabic_df,english_df], ignore_index=True).sample(frac=1).reset_index(drop=True)

Arabic data length: 99999
English data length: 241145
Arabic sample data
       label                                               text
0  Positive  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1  Positive  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2  Positive  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3  Positive  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4  Positive  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...
English sample data
    Unnamed: 0                                            Comment  Sentiment
0           0  lets forget apple pay required brand new iphon...          1
1           1  nz retailers don’t even contactless credit car...          0
2           2  forever acknowledge channel help lessons ideas...          2
3           3  whenever go place doesn’t take apple pay doesn...          0
4           4  apple pay convenient secure easy use used kore...          2

-----------------------------------------------------------

After Pre

In [3]:
print(arabic_df.head())
print(english_df.head())
data.head()

                                                text
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...
                                                text
0       modi salesman century one comes close period
1  life rough enjoy comments videos like one alwa...
2   want namo know namo come har modi ghar ghar mode
3       got twitter friends add none friends twitter
4  along karnataka modi destroy congress democrac...


,text
0,اشكر كل العاملين وخاصة الترجيب كان جدا ممتاز ....
1,البعض يرى أمريكا هى أرض تحقيق الأحلام كأن بها ...
2,استثنائي. الفندق رائع من حيث التعامل والنظافة ...
3,صقر العروبة حقيقة المطعم روووعة وراقي وسيرفيس ...
4,good sir har har modi


Now, to classify this to arabic or enlgish first we should lay our assumptions, if we assume sentences are only of the forms "only english words" and "كلمات عربية فقط" we could make a simple classifier by checking if there exists any letter in either language if not then the its the other language this is the appproach i will be implementing but beolow is anotherr approach worth considering.

the aboce approch is a bit boring so here is a more sophisticated one, we will assume instead the string can have both english and arabic, this will lead to a diffrent solution approach, here is how, first decompose the sentence into word tokens, then check each word, if it contains english letters its "englsih" if it conaints arabic letters its "arabic" if neither than just ignore that word it doesnt matter (its either another language, a punktuation, or somthign else we didnt consider), then we seperate the arabic words into its own string and the english words into its own string, we then run the respectful classifier on each sentence (if it exists) and return the value of the most confident classification.


NO NEED FOR AN ML APPROCH HERE, its not that complicated.

In [9]:
def arabic_or_english(text: str) ->str:
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u08FF')
    english_chars = sum(1 for c in text if c.isascii() and c.isalpha())

    if arabic_chars==0 and english_chars==0:
        return "unknown"

    return "arabic" if arabic_chars>=english_chars else "english"


#tests
def test_arabic_or_english():
    test1 = arabic_or_english("This is an English sentence.")
    test2 = arabic_or_english("هذه عبارة عربية.")
    test3 = arabic_or_english("")
    test4 = arabic_or_english(" ")
    test5 = arabic_or_english("...")
    test6 = arabic_or_english("?")
    test7 = arabic_or_english("hello? this sentence contains punctuation")
    test8 = arabic_or_english("this is a sentence in انجليزي")
    test9 = arabic_or_english("ازيييييييييك يا bro")
    
    
    assert test1=="english"
    assert test2=="arabic"
    assert test3=="unknown"
    assert test4=="unknown"
    assert test5=="unknown"
    assert test6=="unknown"
    assert test7=="english"
    assert test8=="english"
    assert test9=="arabic"

    return True
    
test_arabic_or_english()

True